In [ ]:
import pandas as pd
from pathlib import Path

ROOT = Path.cwd() / "estonia-exam-gap"
RAW = ROOT / "data" / "raw"                # the downloaded Haridussilm / Statistikaamet files (never modified)
EXT = ROOT / "data" / "external"           # cached web look-ups (coordinates, road distances)
PROCESSED = ROOT / "data" / "processed"    # tables this notebook writes


In [ ]:
!rm -rf ROOT
!git clone 'https://github.com/valpar/estonia-exam-gap.git'

fatal: destination path 'estonia-exam-gap' already exists and is not an empty directory.


# X1 : Qualified Teacher Ratio

Ideal Format (Columns)

school_name,
exam_year,
estonian_proficiency_ratio,
teaching_qualification_ratio,

In [87]:
df_raw_qualified_teacher = pd.read_excel(ROOT/RAW/'haridussilm_qualified_teachers_2021-2025_by_school.xlsx', header=2)
df_raw_qualified_teacher.head()

,Indikaator,Õppeasutus/KOV/Eesti,'Date'[Year],Väärtused
0,Eesti keele oskuse nõuetele vastavad õpetajad ...,Abja Gümnaasium,2021,100
1,Eesti keele oskuse nõuetele vastavad õpetajad ...,Abja Gümnaasium,2022,100
2,Eesti keele oskuse nõuetele vastavad õpetajad ...,Abja Gümnaasium,2023,100
3,Eesti keele oskuse nõuetele vastavad õpetajad ...,Abja Gümnaasium,2024,"95,5"
4,Eesti keele oskuse nõuetele vastavad õpetajad ...,Abja Gümnaasium,2025,100


In [ ]:
# Examine the columns
df_raw_qualified_teacher.columns

Index(['Indikaator', 'Õppeasutus/KOV/Eesti', ''Date'[Year]', 'Väärtused'], dtype='object')

In [ ]:
# Translation of estonian column names to english
df_raw_qualified_teacher = df_raw_qualified_teacher.rename(columns={
    "Õppeasutus/KOV/Eesti": "school",
    "'Date'[Year]": "year",
    "Indikaator": "indicator",
    "Väärtused": "percentage"
})
df_raw_qualified_teacher.head()

,indicator,school,year,percentage
0,Eesti keele oskuse nõuetele vastavad õpetajad ...,Abja Gümnaasium,2021,100.0
1,Eesti keele oskuse nõuetele vastavad õpetajad ...,Abja Gümnaasium,2022,100.0
2,Eesti keele oskuse nõuetele vastavad õpetajad ...,Abja Gümnaasium,2023,100.0
3,Eesti keele oskuse nõuetele vastavad õpetajad ...,Abja Gümnaasium,2024,95.5
4,Eesti keele oskuse nõuetele vastavad õpetajad ...,Abja Gümnaasium,2025,100.0


In [ ]:
# Convert percentatge to decimal values
df_raw_qualified_teacher['percentage'] = df_raw_qualified_teacher['percentage'].astype(str).str.replace(',', '.').astype(float)

# Pivot the table to mathcing ideal format definted above
df_raw_qualified_teacher_wide = pd.pivot_table(
    df_raw_qualified_teacher,
    index=['school', 'year'],
    columns='indicator',
    values='percentage'
).reset_index()

# Rename the columns
df_raw_qualified_teacher_wide.columns = [
    'school_name',
    'exam_year',
    'estonian_proficiency_ratio',
    'teaching_qualification_ratio'
]

df_raw_qualified_teacher_wide.head()

,school_name,exam_year,estonian_proficiency_ratio,teaching_qualification_ratio
0,''Eduson'' kool,2021,80.0,100.0
1,''Eduson'' kool,2022,68.8,100.0
2,''Eduson'' kool,2023,60.0,100.0
3,''Eduson'' kool,2024,85.7,100.0
4,''Eduson'' kool,2025,100.0,100.0


X2 : State of the School (State or Municipal)

Ideal Format (Columns)

school_name, ownership, academic_year

In [115]:
# Load schools ran by state
df_raw_school_state = pd.read_excel(ROOT/RAW/'/content/estonia-exam-gap/data/raw/haridussilm_students_2021-2025_state.xlsx', header=2)
df_raw_school_state.info()
df_raw_school_state.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 163 entries, 0 to 162
Data columns (total 4 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   KOV/Õppeasutus   163 non-null    object
 1   Õppeasutus       163 non-null    object
 2   Õppurite arv ap  163 non-null    int64 
 3   Õppeaasta        163 non-null    object
dtypes: int64(1), object(3)
memory usage: 5.2+ KB


/usr/local/lib/python3.13/dist-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Unknown extension is not supported and will be removed
  warn(msg)


,KOV/Õppeasutus,Õppeasutus,Õppurite arv ap,Õppeaasta
0,Haapsalu linn,Läänemaa Ühisgümnaasium,258,2020/21
1,Haapsalu linn,Läänemaa Ühisgümnaasium,272,2021/22
2,Haapsalu linn,Läänemaa Ühisgümnaasium,283,2022/23
3,Haapsalu linn,Läänemaa Ühisgümnaasium,260,2023/24
4,Haapsalu linn,Läänemaa Ühisgümnaasium,268,2024/25


In [116]:
# Load schools ran by municipal
df_raw_school_municipal = pd.read_excel(ROOT/RAW/'/content/estonia-exam-gap/data/raw/haridussilm_students_2021-2025_municipal.xlsx', header=2)
df_raw_school_municipal.info()
df_raw_school_municipal.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 631 entries, 0 to 630
Data columns (total 4 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   KOV/Õppeasutus   631 non-null    object
 1   Õppeasutus       631 non-null    object
 2   Õppurite arv ap  631 non-null    int64 
 3   Õppeaasta        631 non-null    object
dtypes: int64(1), object(3)
memory usage: 19.8+ KB


,KOV/Õppeasutus,Õppeasutus,Õppurite arv ap,Õppeaasta
0,Alutaguse vald,Iisaku Põhikool,48,2020/21
1,Alutaguse vald,Iisaku Põhikool,50,2021/22
2,Alutaguse vald,Iisaku Põhikool,31,2022/23
3,Alutaguse vald,Iisaku Põhikool,20,2023/24
4,Alutaguse vald,Iisaku Põhikool,7,2024/25


In [117]:
# Add ownership label for two different type of school
df_raw_school_state.insert(1, 'ownership', value='state')
df_raw_school_municipal.insert(1, 'ownership', value='municipal')

# Merge school names (Õppeasutus) from both table and give state/municipal labels
df_raw_school_combined = pd.concat([df_raw_school_municipal, df_raw_school_state], ignore_index=True)
df_raw_school_combined.info()
df_raw_school_combined.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 794 entries, 0 to 793
Data columns (total 5 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   KOV/Õppeasutus   794 non-null    object
 1   ownership        794 non-null    object
 2   Õppeasutus       794 non-null    object
 3   Õppurite arv ap  794 non-null    int64 
 4   Õppeaasta        794 non-null    object
dtypes: int64(1), object(4)
memory usage: 31.1+ KB


,KOV/Õppeasutus,ownership,Õppeasutus,Õppurite arv ap,Õppeaasta
0,Alutaguse vald,municipal,Iisaku Põhikool,48,2020/21
1,Alutaguse vald,municipal,Iisaku Põhikool,50,2021/22
2,Alutaguse vald,municipal,Iisaku Põhikool,31,2022/23
3,Alutaguse vald,municipal,Iisaku Põhikool,20,2023/24
4,Alutaguse vald,municipal,Iisaku Põhikool,7,2024/25


In [120]:
# Rename table column propery
# Rename the columns
df_raw_school_combined = df_raw_school_combined[['Õppeasutus', 'ownership','Õppeaasta']]
df_raw_school_combined.columns = [
    'school_name',
    'ownership',
    'academic_year'
]
df_raw_school_combined.head()

,school_name,ownership,academic_year
0,Iisaku Põhikool,municipal,2020/21
1,Iisaku Põhikool,municipal,2021/22
2,Iisaku Põhikool,municipal,2022/23
3,Iisaku Põhikool,municipal,2023/24
4,Iisaku Põhikool,municipal,2024/25




X3 : Distance to Hubs

Ideal Format (Columns)

school_name, distance_to_tallinn, distance_to_tartu

